# AI 드론 데이터 코딩 실습 노트 (실습용)
### Ⅰ. 드론 프로그래밍 기초

이 노트는 교재의 예제를 **직접 채워 넣으며** 익히는 실습용입니다.

1. 셀을 **위에서부터 차례대로** 실행합니다. (**Shift + Enter**)
2. 코드 셀에서 `#` 설명을 읽고 **비어 있는 곳**(`=` 뒤, 설명 아래 빈 줄, 빈 괄호)을 채웁니다.
3. 셀 아래 **실행 결과**와 같게 나오면 성공입니다. 막히면 교재의 같은 번호 예제를 봅니다.
4. `해 보기`는 정답이 하나가 아닙니다. 바꿔 보고, 결과가 왜 달라졌는지 말해 봅시다.

> 빈칸을 채우기 전에 실행하면 빨간 오류가 나옵니다. 고장이 아니라 '아직 덜 썼다'는 뜻입니다.

| 소단원 | 예제 |
|---|---|
| 1-1 드론의 이해와 코딩 환경 구성 | 1-1 |
| 1-2 파이썬 기본 문법과 자료형 | 1-2, 1-3, 1-4, 1-5 |
| 1-3 연산과 입출력 | 1-6, 1-7, 1-8 |
| 1-4 제어문과 자료구조 | 1-9, 1-10, 1-11, 1-12, 1-13, 1-14, 1-15 |
| 1-5 드론 기초 제어 | 1-16, 1-17, 1-18, 1-19, 1-20, 1-21, 1-22, 1-23 |

## 0. 준비하기

아래 셀은 **실습 준비용**입니다. 내용을 이해하지 않아도 괜찮습니다. 실행만 하면 다음 일을 대신 해 줍니다.

- 가상 드론 `drone_sim.py`를 인터넷에서 내려받습니다. 내려받지 못했을 때만 **파일 선택 창**이 뜹니다. 그때는 선생님이 나누어 준 파일을 골라 올립니다.
- 그래프에 한글이 깨지지 않게 해 주는 `font_setup.py`를 만들어 둡니다.

> 코랩은 연결이 끊기면 파일을 잊습니다. 다음 시간에 다시 열었다면 이 셀부터 다시 실행합니다.

In [ ]:
#@title ▶ 준비 셀 (실행만 하세요)
import os, importlib.util

IN_COLAB = importlib.util.find_spec("google.colab") is not None

# 1) 한글 글꼴 설정 모듈(font_setup.py)을 만들어 둔다 (교재 부록 2와 같은 내용)
FONT_SETUP = '''"""font_setup.py - 그래프에 한글이 깨지지 않도록 글꼴을 정해 주는 모듈
그래프를 그리는 예제에서 import font_setup 한 줄만 적으면 된다.
코랩에는 한글 글꼴이 없으므로, 처음 한 번 나눔고딕을 설치한다.
"""
import os
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

NANUM = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 1) 코랩이면 나눔고딕을 설치해서 등록한다
if os.path.exists("/content") and not os.path.exists(NANUM):
    os.system("apt-get -qq install fonts-nanum > /dev/null")
if os.path.exists(NANUM):
    fm.fontManager.addfont(NANUM)

# 2) 운영체제마다 들어 있는 한글 글꼴이 다르므로, 있는 것을 찾아 쓴다
candidates = ["NanumGothic", "Malgun Gothic", "AppleGothic", "Noto Sans CJK KR"]
installed = {f.name for f in fm.fontManager.ttflist}
for name in candidates:
    if name in installed:
        plt.rcParams["font.family"] = name
        break
else:
    print("한글 글꼴을 찾지 못했습니다. 그래프의 한글이 깨질 수 있습니다.")

plt.rcParams["axes.unicode_minus"] = False   # 음수 기호가 깨지지 않도록
'''
if not os.path.exists("font_setup.py"):
    with open("font_setup.py", "w", encoding="utf-8") as f:
        f.write(FONT_SETUP)

# 2) 가상 드론(drone_sim.py)을 내려받는다. 받지 못하면 파일 선택 창을 띄운다
import urllib.request
URL = "https://raw.githubusercontent.com/ralralra/dronebook/main/data/drone_sim.py"
try:
    urllib.request.urlretrieve(URL, "drone_sim.py")
except Exception:
    print("drone_sim.py 를 내려받지 못했습니다.")
if not os.path.exists("drone_sim.py") and IN_COLAB:
    from google.colab import files
    print("선생님이 나누어 준 drone_sim.py 를 골라 올려 주세요.")
    files.upload()

for name in ["drone_sim.py", "font_setup.py"]:
    print(("✓ 준비됨  " if os.path.exists(name) else "✗ 없음    ") + name)


# 1-1 드론의 이해와 코딩 환경 구성

**학습 목표**

- 드론의 주요 부품이 하는 일을 설명할 수 있습니다.
- 드론이 뜨고 움직이는 데 작용하는 네 가지 힘을 설명할 수 있습니다.
- 드론 운용 안전수칙을 알고 점검표를 작성할 수 있습니다.
- 코랩에 코딩 환경을 구성하고 가상 드론에 연결할 수 있습니다.

<sub>관련 성취기준 [드코1 01-01-01] / [드코1 01-01-02]</sub>

> **생각 열기**  
> 택배 상자를 실은 드론이 아파트 옥상으로 날아갑니다. 산불이 난 곳에서는 드론이 먼저 현장을 살핍니다. 이 비행체는 날개를 퍼덕이지도 않고 활주로를 달리지도 않는데 어떻게 공중에 떠 있는 것일까요? 그리고 조종기를 잡은 사람이 없을 때 드론은 무엇을 보고 움직이는 것일까요?

<sub>교재에서 읽기: ■ 1. 드론이란 무엇인가 · ■ 2. 드론의 구성 요소 · ■ 3. 드론에 작용하는 네 가지 힘 · ■ 4. 자세 제어의 원리</sub>

## ■ 5. 드론 운용 안전수칙

드론의 프로펠러는 매우 빠르게 돌기 때문에 손이나 얼굴에 닿으면 다칠 수 있습니다. 프로그램이 잘못되면 기체가 예상하지 못한 방향으로 날아갈 수도 있습니다. 그래서 실기체를 다루기 전에는 반드시 다음 수칙을 확인합니다.

> **안전 수칙 · 실기체 비행 전 반드시 지킬 것**  
> 1. 교사의 지도 아래, 정해진 실습 구역에서만 비행합니다.
> 2. 비행 전 프로펠러의 파손 여부와 배터리 잔량을 확인합니다.
> 3. 사람과 최소 3 m 이상 거리를 두고, 사람을 향해 비행하지 않습니다.
> 4. 비상 정지 방법(조종기 L1을 누른 채 왼쪽 스틱을 아래로)을 미리 익히고, 문제가 생기면 즉시 정지시킵니다.
> 5. 가상 드론에서 충분히 검증한 프로그램만 실기체에서 실행합니다.
> 6. 실내에서는 선풍기·에어컨 바람과 커튼 등 주변 환경을 확인합니다.

### ====== 해 보기 ======

**우리 반 비행 전 점검표 만들기**

모둠별로 위의 안전수칙을 바탕으로 '비행 전 점검표'를 만들어 봅시다. 점검 항목은 눈으로 직접 확인할 수 있는 것이어야 합니다. (예: '프로펠러에 금이 가 있지 않은가?' → 확인 가능 / '안전하게 비행할 것인가?' → 확인 어려움)  
모둠에서 만든 점검표를 다른 모둠의 것과 비교해 보고, 빠진 항목을 서로 채워 줍니다.

## ■ 6. 코랩에 코딩 환경 만들기

이 교과서의 예제는 모두 파이썬(Python)으로 작성합니다. 파이썬은 문법이 간결해 처음 배우기 쉽고, 데이터 분석과 인공지능 분야에서도 널리 쓰입니다. 프로그램은 구글 코랩(Colab)에서 작성합니다. 코랩은 웹 브라우저만 있으면 설치 없이 파이썬을 실행할 수 있습니다.

1. colab.research.google.com에 구글 계정으로 로그인하고 새 노트를 만듭니다.
2. 첫 코드 셀에서 준비 셀을 실행합니다. 이 노트에서는 맨 위 '0. 준비하기'의 셀이 준비 셀입니다.
3. 그 아래 코드 셀에 프로그램을 입력하고 실행 단추(또는 Shift + Enter)를 누릅니다.

준비 셀은 이 책에서 쓰는 drone\_sim.py와 font\_setup.py를 마련해 줍니다. 실행한 뒤 왼쪽의 파일(폴더 아이콘)을 누르면 두 파일이 보입니다.

drone\_sim.py는 가상 드론입니다. 실제 기체가 없어도 프로그램이 제대로 동작하는지 확인할 수 있고, 비행 기록을 파일로 저장해 주기 때문에 Ⅱ단원의 데이터 분석에도 씁니다. 함수 이름은 실제 코드론에듀 라이브러리와 같게 만들었습니다. font\_setup.py는 그래프의 한글이 깨지지 않게 해 주는 파일로, Ⅲ단원에서 씁니다.

> **짚고 가기 · 다음 시간에는 준비 셀부터**  
> 코랩은 구글 서버를 잠깐 빌려 쓰는 것이라, 한동안 쓰지 않아 연결(런타임)이 끊기면 내려받은 파일이 사라집니다. 노트를 다시 열었다면 준비 셀부터 다시 실행합니다. 준비 셀을 빠뜨리고 예제를 실행하면 ModuleNotFoundError: No module named 'drone\_sim'이 나옵니다. 'drone\_sim이라는 파일을 찾지 못했다'는 뜻입니다.

**예제 1-1. 가상 드론에 연결하기**

In [ ]:
# 예제 1-1. 가상 드론에 연결하기
# 가상 드론 불러오기


drone =        # 드론 객체 만들기
# 연결하기

print("배터리 잔량:", drone.get_battery(), "%")
# 연결 닫기


실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
배터리 잔량: 100 %
[가상] 연결을 닫았습니다.
```

첫 줄의 from drone\_sim import \*는 'drone\_sim 파일에 있는 것을 모두 가져오라'는 뜻입니다. Drone()으로 드론을 하나 만들어 drone이라는 이름을 붙였고, 이제부터는 drone. 뒤에 명령을 붙여 드론을 다룹니다. [가상]으로 시작하는 줄은 가상 드론이 지금 무엇을 했는지 알려 주는 안내문입니다.

> **짚고 가기 · 실제 드론으로 바꿀 때**  
> 가상 드론의 함수 이름은 코드론에듀 라이브러리와 같습니다. 그래서 실기체로 날릴 때는 첫 줄 하나만 from codrone\_edu.drone import \*로 바꾸면 됩니다. 다만 코랩은 학생 PC에 꽂힌 조종기를 볼 수 없으므로, 실기체 비행은 1-5에서 파이참으로 합니다.

<sub>교재에서 읽기: ■ 7. 이 단원에서 쓸 명령</sub>

### 핵심 정리

- 드론은 사람이 타지 않고 원격 조종이나 프로그램으로 비행하는 무인 비행 장치입니다.
- 드론에는 양력·중력·추력·항력이 작용하며, 양력과 중력의 크기를 조절해 뜨고 내립니다.
- 비행 제어기는 우리가 내린 명령을 네 모터의 회전 속도로 바꾸어 줍니다.
- 실기체를 다루기 전에는 안전수칙을 확인하고, 가상 드론에서 검증한 프로그램만 실행합니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 1-2 파이썬 기본 문법과 자료형

**학습 목표**

- 변수의 개념을 이해하고 알맞은 이름을 붙일 수 있습니다.
- 정수, 실수, 문자열, 논리형의 특징을 구분할 수 있습니다.
- 드론의 상태 정보를 변수에 저장하고 보기 좋게 출력할 수 있습니다.

<sub>관련 성취기준 [드코1 01-02-01] / [드코1 01-02-02]</sub>

> **생각 열기**  
> 드론을 띄우기 전에 우리는 배터리가 얼마나 남았는지, 지금 높이가 몇인지 확인합니다. 이렇게 수시로 바뀌는 값을 프로그램은 어디에 담아 둘까요? 그리고 '87'이라는 숫자와 '87%'라는 글자는 컴퓨터에게 같은 것일까요?

## ■ 1. 변수: 값에 붙이는 이름표

변수(variable)는 값에 붙이는 이름입니다. 비행 중에 바뀌는 배터리 잔량이나 높이처럼 달라지는 정보를 다룰 때 씁니다. 파이썬에서는 = 기호로 오른쪽의 값에 왼쪽의 이름을 붙입니다.

**예제 1-2. 변수에 값 저장하기**

In [ ]:
# 예제 1-2. 변수에 값 저장하기
battery =        # 배터리 잔량
height =        # 높이
# battery 출력
print()
# height 출력
print()

실행 결과

```
87
120
```

> **짚고 가기 · = 는 '같다'가 아닙니다.**  
> 수학에서 = 는 양쪽이 같다는 뜻이지만, 파이썬에서 = 는 '오른쪽 값에 이 이름을 붙여라'는 명령입니다. 그래서 battery = battery - 5 같은 문장도 성립합니다. 오른쪽을 먼저 계산해 82를 만들고, battery라는 이름표를 87에서 떼어 82에 다시 붙이는 것입니다.

> **한 걸음 더 · 상자가 아니라 이름표인 까닭**  
> 변수를 '값을 담는 상자'로 설명하는 책도 많습니다. 하지만 파이썬에서는 값이 따로 있고 변수는 그 값에 붙은 이름표입니다. 하나의 값에 이름표를 여러 개 붙일 수도 있습니다. 이 차이는 1-4에서 리스트를 배울 때 다시 중요해집니다.

<sub>교재에서 읽기: ■ 2. 변수 이름 짓기</sub>

## ■ 3. 기본 자료형

파이썬은 값의 종류에 따라 자료형(data type)을 구분합니다. 자료형이 다르면 할 수 있는 일도 달라지므로, 값의 성격에 맞는 자료형을 써야 합니다.

| 자료형 | 이름 | 저장하는 값 | 드론에서의 예 |
|---|---|---|---|
| 정수 | int | 소수점이 없는 수 | 배터리 87, 높이 120 |
| 실수 | float | 소수점이 있는 수 | 속도 1.5, 비행 시간 12.4 |
| 문자열 | str | 따옴표로 감싼 글자 | 기체 이름 "드론-1호" |
| 논리형 | bool | 참(True) 또는 거짓(False) | 비행 중인가? True |

<sub>표 1-5. 파이썬의 기본 자료형</sub>

**예제 1-3. 여러 자료형으로 드론 정보 저장하기**

In [ ]:
# 예제 1-3. 여러 자료형으로 드론 정보 저장하기
name =        # 문자열
battery =        # 정수
speed =        # 실수
is_flying =        # 논리형

# type()으로 네 변수의 자료형을 차례로 확인하기
print(type())
print(type())
print(type())
print(type())

실행 결과

```
<class 'str'>
<class 'int'>
<class 'float'>
<class 'bool'>
```

type()은 변수가 가리키는 값이 어떤 자료형인지 알려 주는 함수입니다. 결과에 나오는 str, int, float, bool이 각각 문자열, 정수, 실수, 논리형입니다. class라는 말이 보이는데, 파이썬에서는 숫자 하나도 틀(class)에서 찍어 낸 객체이기 때문입니다. 1-1에서 만든 drone도 Drone이라는 틀에서 찍어 낸 객체입니다.

> **한 걸음 더 · 따옴표가 있고 없고의 차이**  
> battery = 87은 숫자 87이지만 battery = "87"은 문자 '87'입니다. 숫자는 더하거나 뺄 수 있지만 문자는 그렇지 않습니다. 실제로 "87" - 5를 실행하면 TypeError: unsupported operand type(s) for -: 'str' and 'int' 오류가 납니다. '문자열과 정수는 뺄셈을 할 수 없다'는 뜻입니다. 화면에 보이는 모습이 같아도 컴퓨터에게는 전혀 다른 값입니다.

## ■ 4. 자료형 바꾸기

필요하면 자료형을 바꿀 수 있습니다. 이를 형 변환이라고 합니다.

**예제 1-4. 형 변환**

In [ ]:
# 예제 1-4. 형 변환
battery_text = "87"              # 글자 87
battery =        # 숫자로 바꾸기
print(battery + 5)

height = 120
print("고도: " +  + " cm")   # 숫자를 글자로 바꿔 이어 붙이기

실행 결과

```
92
고도: 120 cm
```

## ■ 5. 드론 상태 표시 프로그램 만들기

이제 배운 내용을 모아 드론의 상태를 한눈에 보여 주는 프로그램을 만들어 봅시다. 여러 값을 섞어 출력할 때는 f-문자열을 쓰면 편리합니다. 문자열 앞에 f를 붙이고 중괄호 { } 안에 변수 이름을 적으면 그 자리에 값이 들어갑니다.

**예제 1-5. 드론 상태 표시판**

In [ ]:
# 예제 1-5. 드론 상태 표시판
from drone_sim import *

drone = Drone()
drone.pair()

name = "드론-1호"
# 드론에게 배터리 잔량 묻기
battery =
# 드론에게 지금 높이 묻기
height =
is_flying = False

print("=" * 28)
print(f" 기체 이름 : { }")
print(f" 배터리    : { } %")
print(f" 높이      : { } cm")
print(f" 비행 중   : { }")
print("=" * 28)
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
============================
 기체 이름 : 드론-1호
 배터리    : 100 %
 높이      : 0.0 cm
 비행 중   : False
============================
[가상] 연결을 닫았습니다.
```

"=" \* 28은 = 문자를 28번 반복해 선을 긋는 방법입니다. 문자열에 숫자를 곱하면 그 횟수만큼 반복됩니다. 높이가 0.0인 까닭은 아직 이륙하지 않아 드론이 바닥에 있기 때문입니다. 배터리는 정수(100)로, 높이는 실수(0.0)로 나왔습니다. get\_battery()와 get\_height()가 돌려주는 자료형이 다르기 때문입니다.

### ====== 해 보기 ======

**나만의 상태 표시판 꾸미기**

1. 예제 1-5를 실행해 보고, 기체 이름을 자신이 정한 이름으로 바꿔 봅시다.
2. '속도'와 '비행 가능 여부' 항목을 더해 봅시다. 각각 어떤 자료형이 알맞을까요?
3. 선을 긋는 문자를 - 나 \*로 바꾸고, 길이도 조절해 봅시다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 변수는 값에 붙이는 이름표이며, = 는 오른쪽 값에 왼쪽 이름을 붙이라는 뜻입니다.
- 기본 자료형에는 정수(int), 실수(float), 문자열(str), 논리형(bool)이 있습니다.
- type()으로 자료형을 확인하고, int()·str()로 자료형을 바꿀 수 있습니다.
- f-문자열을 쓰면 글자와 변수의 값을 섞어 보기 좋게 출력할 수 있습니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 1-3 연산과 입출력

**학습 목표**

- 산술·비교·논리 연산자로 값을 계산하고 판단할 수 있습니다.
- input()으로 값을 입력받아 계산에 쓸 수 있습니다.
- 비행 거리와 시간, 배터리 소모량을 계산하고 그 결과를 해석할 수 있습니다.

<sub>관련 성취기준 [드코1 01-03-01] / [드코1 01-03-02]</sub>

> **생각 열기**  
> '이 드론으로 90 m 떨어진 곳까지 다녀올 수 있을까?' 이 질문에 답하려면 속도로 시간을 구하고, 시간으로 배터리 소모량을 구해야 합니다. 계산이 곧 판단의 근거가 되는 것입니다.

<sub>교재에서 읽기: ■ 1. 산술 연산자</sub>

## ■ 2. 비행 시간과 배터리 계산하기

드론의 비행 시간과 배터리 소모량은 사양서의 숫자로 예측할 수 있습니다. 코드론에듀는 한 번 충전으로 7\~8분을 납니다(표 1-1). 가운데값인 7.5분을 날며 100%를 쓴다고 보면, 1초에 몇 %를 쓰는지 계산할 수 있습니다.

**예제 1-6. 비행 시간과 배터리 소모량 계산**

In [ ]:
# 예제 1-6. 비행 시간과 배터리 소모량 계산
distance = 300                         # 비행 거리(cm)
speed = 50                             # 속도(cm/s)
flight_min = 7.5                       # 한 번 충전으로 나는 시간(분)
use_per_sec =        # 1초당 배터리 소모량(%)

time =        # 비행 시간(초)
used =        # 쓰는 배터리(%)
left =        # 남는 배터리(%)

print(f"1초당 소모 : {use_per_sec} %")
print(f"비행 시간 : {time} 초")
print(f"배터리 소모 : {used} %")
print(f"남는 배터리 : {left} %")

실행 결과

```
1초당 소모 : 0.2222222222222222 %
비행 시간 : 6.0 초
배터리 소모 : 1.3333333333333333 %
남는 배터리 : 98.66666666666667 %
```

괄호가 계산 순서를 정합니다. 100 / (7.5 \* 60)은 괄호 안의 7.5 × 60 = 450초를 먼저 구하고 100을 그것으로 나눕니다. 괄호를 빼고 100 / 7.5 \* 60이라고 쓰면 왼쪽부터 계산해 800이 되는데, 이는 전혀 다른 뜻입니다.

다만 이 계산은 제자리 비행 기준입니다. 빠르게 움직이거나 바람이 불면 배터리는 더 빨리 줄어듭니다. 계산에 쓴 가정을 함께 적어 두어야 결과를 올바르게 읽을 수 있습니다.

> **짚고 가기 · 자릿수를 정리하는 round()**  
> 1.3333333333333333처럼 긴 수는 읽기 어렵습니다. round(값, 자릿수)로 정리합니다. round(used, 1)은 1.3, round(left, 1)은 98.7이 됩니다.  
> 실수 계산에는 이상한 꼬리가 붙기도 합니다. print(0.1 + 0.2)를 실행하면 0.3이 아니라 0.30000000000000004가 나옵니다. 오류가 아니라, 컴퓨터가 실수를 2진수로 저장하면서 생기는 아주 작은 오차입니다. 0.1이나 0.2는 2진수로 딱 떨어지지 않기 때문입니다. 화면이나 보고서에 숫자를 나타낼 때는 round()로 자릿수를 정리하는 습관을 들입니다.

## ■ 3. 값을 입력받기

input()을 쓰면 프로그램을 실행하는 사람이 값을 직접 넣을 수 있습니다. 코랩에서는 셀 아래에 입력 칸이 나타납니다. 주의할 점은 input()으로 받은 값은 항상 문자열이라는 것입니다. 계산에 쓰려면 int()나 float()로 바꿔야 합니다.

**예제 1-7. 거리를 입력받아 비행 시간 계산하기**

In [ ]:
# 예제 1-7. 거리를 입력받아 비행 시간 계산하기
# input()으로 받은 글자를 float()으로 실수로 바꾸기
distance =
speed = 50
# 시간 = 거리 ÷ 속도
time =
print(f"예상 비행 시간은 {round(time, 1)} 초입니다.")

실행 결과

```
비행 거리를 입력하세요(cm): 450
예상 비행 시간은 9.0 초입니다.
```

> **짚고 가기 · 자주 생기는 오류**  
> float()를 빠뜨리고 input()만 쓰면 "450" / 50을 계산하게 되어 TypeError: unsupported operand type(s) for /: 'str' and 'int' 오류가 납니다. 오류 문구에 str과 int가 함께 보이면 자료형을 바꾸지 않았는지 먼저 확인합니다.

### ====== 해 보기 ======

**드론 택배의 배송 시간 구하기**

드론 택배 회사의 드론이 물류 센터를 떠나 A 지점에 물건을 내려 주고, C 지점과 B 지점에 차례로 들른 뒤 다시 A 지점으로 돌아옵니다. 구간마다 날아야 할 거리는 다음과 같습니다. 모두 몇 분이 걸리는지 계산하는 프로그램을 만들어 봅시다.

- 물류 센터 → A 지점: 5 km
- A 지점 → C 지점: 3.4 km
- C 지점 → B 지점: 2.6 km
- B 지점 → A 지점: 4.2 km

1. 네 구간의 거리를 변수 네 개에 담고, 모두 더해 총 비행 거리를 구합니다.
2. input()으로 드론의 속도를 입력받습니다. 단위는 km/h입니다.
3. 총 거리를 속도로 나누면 걸리는 시간이 '시간' 단위로 나옵니다. 60을 곱해 '분'으로 바꾸고, round()로 소수 첫째 자리까지 출력합니다.
4. 속도에 36을 넣어 실행합니다. 25.3분이 나오면 바르게 만든 것입니다.
5. A, C, B 세 지점에서 물건을 내리는 데 2분씩 걸린다면 모두 몇 분이 걸릴까요? 프로그램에 한 줄을 더해 구해 봅니다.
6. 속도를 50으로 바꾸면 비행 시간은 몇 분 줄어드나요? 속도를 높이면 좋기만 할지, 예제 1-6에서 배운 배터리와 함께 생각해 봅시다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.

## ■ 4. 비교 연산자와 논리 연산자

비교 연산자는 두 값을 견주어 참(True) 또는 거짓(False)을 알려 줍니다. 비행 가능 여부처럼 판단이 필요한 곳에 씁니다.

| 연산자 | 뜻 | 예 | 결과 |
|---|---|---|---|
| == | 같다 | battery == 100 | False |
| != | 같지 않다 | battery != 100 | True |
| &gt; &lt; | 크다 / 작다 | battery &gt; 20 | True |
| &gt;= &lt;= | 크거나 같다 / 작거나 같다 | battery &gt;= 87 | True |

<sub>표 1-7. 비교 연산자 (battery = 87일 때)</sub>

논리 연산자는 여러 조건을 묶을 때 씁니다. and는 모두 참일 때, or는 하나라도 참일 때 참이 되며, not은 참과 거짓을 뒤집습니다.

**예제 1-8. 비행 가능 여부 판단하기**

In [ ]:
# 예제 1-8. 비행 가능 여부 판단하기
battery = 87     # 배터리 잔량(%)
wind = 3.2       # 풍속(m/s)

# 배터리가 30 이상인가?
battery_ok =
# 풍속이 5 이하인가?
wind_ok =
can_fly =        # 둘 다 만족해야 비행 가능

print("배터리 조건:", battery_ok)
print("바람 조건  :", wind_ok)
print("비행 가능  :", can_fly)

실행 결과

```
배터리 조건: True
바람 조건  : True
비행 가능  : True
```

조건마다 변수를 따로 두면, 비행이 불가능할 때 어떤 조건 때문인지 바로 알 수 있습니다. 이것이 다음 소단원에서 배울 조건문의 바탕이 됩니다. 여기서 쓴 30%와 5 m/s는 우리 반이 정한 기준입니다. 기준을 숫자로 적어 두면 누가 판단해도 같은 결론이 나옵니다.

### ====== 해 보기 ======

**배송 가능 거리 계산기**

배터리를 100%에서 출발해 20%까지만 쓴다고 할 때, 왕복으로 갈 수 있는 최대 거리를 계산하는 프로그램을 만들어 봅시다.

- 속도와 1초당 배터리 소모량은 예제 1-6의 값을 씁니다.
- 쓸 수 있는 배터리는 몇 %인가요? 그것으로 몇 초를 날 수 있나요?
- 왕복이므로 편도 거리는 전체 거리의 절반입니다.
- 계산한 편도 거리를 코드론에듀의 통신 거리 50 m(표 1-1)와 비교해 봅시다. 배터리만 보고 계획을 세우면 어떤 일이 생길까요? 실제 배송에 쓰려면 무엇을 더 고려해야 할지 모둠에서 이야기해 봅시다.

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 산술 연산자로 값을 계산하며, /는 소수점까지, //는 몫만 구합니다.
- input()으로 받은 값은 문자열이므로 계산 전에 int()나 float()로 바꿉니다.
- 비교 연산자는 참과 거짓을 알려 주고, 논리 연산자(and, or, not)로 조건을 묶습니다.
- 계산 결과는 그 자체로 끝나지 않고 비행 가능 여부를 판단하는 근거가 됩니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 1-4 제어문과 자료구조

**학습 목표**

- 조건문으로 상황에 따라 다르게 동작하는 프로그램을 작성할 수 있습니다.
- 반복문으로 같은 동작을 여러 번 수행하는 프로그램을 작성할 수 있습니다.
- 리스트·튜플·딕셔너리의 특성을 구분하고 경유지와 비행 기록을 저장할 수 있습니다.

<sub>관련 성취기준 [드코1 01-04-01] / [드코1 01-04-02]</sub>

> **생각 열기**  
> 지금까지 만든 프로그램은 위에서 아래로 한 번만 실행되었습니다. 그런데 실제 비행에서는 '배터리가 부족하면 돌아와야' 하고, '네 모서리를 돌 때까지 같은 동작을 반복해야' 합니다. 프로그램의 흐름을 바꾸려면 무엇이 필요할까요?

<sub>교재에서 읽기: ■ 1. 프로그램의 세 가지 흐름</sub>

## ■ 2. 조건문 if

조건문은 조건이 참일 때만 특정 명령을 실행하게 합니다. 조건 뒤에는 반드시 콜론(:)을 붙이고, 조건에 해당할 때 실행할 문장은 들여쓰기(보통 공백 4칸)해서 씁니다.

**예제 1-9. 배터리 경고 출력하기**

In [ ]:
# 예제 1-9. 배터리 경고 출력하기
battery = 15

# 만약 배터리가 20보다 작으면

    print("경고: 배터리가 부족합니다. 즉시 착륙하세요.")
# 그렇지 않으면

    print("정상: 비행을 계속합니다.")

실행 결과

```
경고: 배터리가 부족합니다. 즉시 착륙하세요.
```

조건이 셋 이상일 때는 elif를 씁니다. 위에서부터 차례로 검사해 처음으로 참이 되는 곳만 실행하고 나머지는 건너뜁니다.

**예제 1-10. 배터리 상태를 단계로 나누기**

In [ ]:
# 예제 1-10. 배터리 상태를 단계로 나누기
battery = 45

# 만약 배터리가 70 이상이면

    print("충분: 계획한 임무를 모두 수행할 수 있습니다.")
# 그렇지 않고 30 이상이면 (elif)

    print("보통: 짧은 거리만 비행하세요.")
# 그렇지 않고 20 이상이면 (elif)

    print("주의: 복귀를 준비하세요.")
# 어디에도 해당하지 않으면

    print("위험: 이륙하지 마세요.")

실행 결과

```
보통: 짧은 거리만 비행하세요.
```

45는 30 이상이면서 20 이상이기도 합니다. 그런데도 '보통'만 출력된 까닭은, 처음 참이 된 곳에서 멈추기 때문입니다. 그래서 elif는 기준이 큰 것부터 순서대로 씁니다.

> **짚고 가기 · 들여쓰기가 문법입니다.**  
> 파이썬은 들여쓰기로 명령이 어디에 속하는지 구분합니다. 들여쓰기가 어긋나면 IndentationError가 나거나, 오류 없이 의도와 다르게 동작합니다. 한 파일 안에서는 들여쓰기 칸 수를 똑같이 맞춥니다. 코랩은 콜론 다음 줄에서 자동으로 들여 씁니다.

## ■ 3. 반복문 for

같은 동작을 정해진 횟수만큼 되풀이할 때는 for를 씁니다. range(4)는 0, 1, 2, 3의 네 값을 차례로 내어 주므로 네 번 반복합니다.

**예제 1-11. 사각형 경로 비행**

In [ ]:
# 예제 1-11. 사각형 경로 비행
from drone_sim import *

drone = Drone()
drone.pair()
drone.takeoff()

# 네 번 반복

    # 앞으로 100 cm
    
    # 오른쪽으로 90도
    

print("최종 위치:", drone.get_pos_x(), drone.get_pos_y())
drone.land()
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 100cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 100cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 100cm
[가상] 오른쪽으로 90도 회전
[가상] 앞으로 100cm
[가상] 오른쪽으로 90도 회전
최종 위치: 0.0 -0.0
[가상] 착륙
[가상] 연결을 닫았습니다.
```

100 cm 전진한 뒤 90도 회전하기를 네 번 반복하면 정사각형을 그리며 제자리로 돌아옵니다. 최종 위치가 0인 것으로 확인할 수 있습니다. -0.0은 0과 같은 값입니다. 방향을 계산하는 과정에서 아주 작은 음수가 생겨 부호만 남은 것입니다. 반복문을 쓰지 않으면 같은 두 줄을 네 번 써야 하지만, 반복문을 쓰면 횟수만 바꿔 삼각형이나 육각형도 그릴 수 있습니다.

## ■ 4. 반복문 while

횟수가 정해져 있지 않고 어떤 조건을 만족하는 동안 반복할 때는 while을 씁니다.

**예제 1-12. 배터리가 충분한 동안 비행하기**

In [ ]:
# 예제 1-12. 배터리가 충분한 동안 비행하기
from drone_sim import *

drone = Drone()
drone.sim_set_battery(32)          # 가상 전용: 배터리를 32 % 로 맞춰 둔다
drone.pair()
drone.takeoff()

# 배터리가 30 % 이상인 동안

    # 앞으로 100 cm
    
    print("  남은 배터리:", drone.get_battery(), "%")

print("배터리가 30 % 아래로 내려가 착륙합니다.")
drone.land()
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 32 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 100cm
  남은 배터리: 30 %
[가상] 앞으로 100cm
  남은 배터리: 30 %
[가상] 앞으로 100cm
  남은 배터리: 29 %
배터리가 30 % 아래로 내려가 착륙합니다.
[가상] 착륙
[가상] 연결을 닫았습니다.
```

몇 번 반복할지 코드에는 어디에도 적혀 있지 않습니다. 반복할 때마다 배터리를 다시 읽고, 30 아래로 내려간 순간 반복을 끝냅니다. 배터리가 더 남아 있었다면 더 오래 날았을 것입니다.

> **안전 수칙 · 끝나지 않는 반복은 위험합니다.**  
> while문의 조건이 영원히 참이면 프로그램이 멈추지 않습니다. 이를 무한 반복이라고 합니다. 가상 드론에서는 화면만 멈추지만, 실기체에서는 드론이 계속 날아가 사고로 이어질 수 있습니다. while을 쓸 때는 반복을 끝내는 조건이 반드시 언젠가 만족되는지 확인합니다. 코랩에서 무한 반복에 빠지면 셀 왼쪽의 정지 단추를 누릅니다.

## ■ 5. 리스트: 여러 값을 순서대로

리스트(list)는 여러 값을 순서대로 묶는 자료구조입니다. 대괄호 [ ]로 만들고, 0번부터 시작하는 번호(인덱스)로 값을 꺼냅니다.

**예제 1-13. 경유지 목록 다루기**

In [ ]:
# 예제 1-13. 경유지 목록 다루기
points = ["창고", "공원", "학교", "병원"]

print("경유지 수:", )
print("첫 번째 :", )
print("마지막   :", )

# 맨 뒤에 추가

print("추가 후  :", points)

실행 결과

```
경유지 수: 4
첫 번째 : 창고
마지막   : 병원
추가 후  : ['창고', '공원', '학교', '병원', '소방서']
```

> **짚고 가기 · 이름표가 두 개인 리스트**  
> 1-2에서 변수는 상자가 아니라 이름표라고 했습니다. 예제 1-13 뒤에 route = points를 쓰고 route.append("우체국")을 실행해 봅시다. route에만 더했는데 print(points)를 해 보면 마지막에 '우체국'이 붙어 있습니다. route = points는 리스트를 복사한 것이 아니라, 같은 리스트에 이름표를 하나 더 붙인 것이기 때문입니다.

리스트는 for와 함께 쓸 때 힘을 발휘합니다. 값을 하나씩 꺼내 같은 처리를 할 수 있습니다.

**예제 1-14. 구간을 순서대로 비행하기**

In [ ]:
# 예제 1-14. 구간을 순서대로 비행하기
from drone_sim import *

plan = [100, 150, 80]        # 각 구간의 비행 거리(cm)

drone = Drone()
drone.pair()
drone.takeoff()
# plan에서 거리를 하나씩 꺼내며 반복

    # 꺼낸 거리만큼 앞으로
    
    drone.turn_right(120)
drone.land()
drone.close()
print("비행 구간 수:", len(plan))

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 100cm
[가상] 오른쪽으로 120도 회전
[가상] 앞으로 150cm
[가상] 오른쪽으로 120도 회전
[가상] 앞으로 80cm
[가상] 오른쪽으로 120도 회전
[가상] 착륙
[가상] 연결을 닫았습니다.
비행 구간 수: 3
```

## ■ 6. 튜플과 딕셔너리

튜플(tuple)은 소괄호 ( )로 만들며 리스트와 비슷하지만, 한 번 만들면 값을 바꿀 수 없습니다. 좌표처럼 바뀌면 안 되는 값을 묶을 때 알맞습니다. point = (100, 50)을 만든 뒤 point[0] = 10을 실행하면 TypeError: 'tuple' object does not support item assignment 오류가 납니다. 실수로 좌표가 바뀌는 일을 파이썬이 막아 주는 것입니다.

딕셔너리(dictionary)는 중괄호 { }로 만들며, 순서 대신 이름(키)으로 값을 찾습니다. 비행 기록처럼 항목마다 이름이 있는 자료에 알맞습니다.

**예제 1-15. 비행 기록을 딕셔너리로 저장하기**

In [ ]:
# 예제 1-15. 비행 기록을 딕셔너리로 저장하기
record = {"시간": 12.5, "고도": 120, "배터리": 87, "위치": (100, 50)}

print("고도:", , "cm")
# 값 바꾸기

# 항목 추가


# record의 키를 하나씩 꺼내며 반복

    print(f"{key} : {record[key]}")

실행 결과

```
고도: 120 cm
시간 : 12.5
고도 : 120
배터리 : 85
위치 : (100, 50)
풍속 : 3.2
```

딕셔너리는 Ⅱ단원에서 만날 데이터프레임의 한 줄과 같은 모양입니다. 키가 열 이름이 되고, 값이 그 줄의 칸이 됩니다.

| 자료구조 | 기호 | 특징 | 드론에서의 쓰임 |
|---|---|---|---|
| 리스트 | [ ] | 순서가 있고 값을 바꿀 수 있습니다 | 경유지 목록, 구간별 거리 |
| 튜플 | ( ) | 순서가 있으나 값을 바꿀 수 없습니다 | 좌표 (x, y) |
| 딕셔너리 | { } | 이름(키)으로 값을 찾습니다 | 비행 기록, 기체 정보 |

<sub>표 1-8. 세 가지 자료구조의 비교</sub>

### ====== 해 보기 ======

**안전한 비행 계획 프로그램**

구간별 거리를 담은 리스트와 조건문을 함께 써서, 배터리가 기준 아래로 떨어지면 남은 구간을 포기하고 착륙하는 프로그램을 만들어 봅시다.

- 처음에 drone.sim\_set\_battery()로 배터리를 낮게 맞춰 둡니다.
- for로 구간을 돌면서 매번 drone.get\_battery()로 배터리를 확인합니다.
- 조건에 걸리면 break로 반복을 빠져나온 뒤 착륙합니다.
- 몇 번째 구간까지 갔는지 함께 출력해 봅시다. (1-5의 예제 1-20과 비교해 보세요.)

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


### 핵심 정리

- 프로그램의 흐름에는 순차, 선택(조건문), 반복(반복문)이 있습니다.
- if \~ elif \~ else로 상황을 나누고, 들여쓰기로 소속을 구분합니다.
- 횟수가 정해졌으면 for, 조건이 만족되는 동안이면 while을 씁니다. 무한 반복에 주의합니다.
- 리스트는 순서대로, 튜플은 바뀌지 않게, 딕셔너리는 이름으로 값을 다룹니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# 1-5 드론 기초 제어

**학습 목표**

- 드론 라이브러리의 함수를 불러 연결·이륙·이동·회전·착륙을 순서대로 실행할 수 있습니다.
- 인자(거리·단위·속도·각도)를 바꾸어 비행을 조절하고, 반환값으로 드론의 상태를 확인할 수 있습니다.
- 되풀이되는 비행 동작을 함수로 묶어 다시 쓸 수 있습니다.
- 파이참에서 실기체를 페어링하고, 안전수칙을 지키며 기본 동작을 제어할 수 있습니다.

<sub>관련 성취기준 [드코1 01-05-01] / [드코1 01-05-02]</sub>

> **생각 열기**  
> 우리는 이미 함수를 써 왔습니다. print("안녕")은 화면에 글자를 띄우라는 명령이고, len(plan)은 리스트의 길이를 알려 달라는 명령입니다. print가 안에서 어떻게 글자를 그리는지 몰라도 우리는 잘 썼습니다. 이름과 괄호에 넣을 값만 알면 되기 때문입니다.

드론도 마찬가지입니다. 드론을 띄우려면 네 모터의 회전 속도를 쉬지 않고 맞춰야 합니다. 그 복잡한 일을 누군가 takeoff()라는 이름 안에 담아 두었다면, 우리는 이름 하나만 부르면 됩니다. 코드론에듀 드론 객체가 가진 공개 함수는 263개입니다. 이 가운데 어떤 것을, 어떤 값과 함께, 어떤 순서로 부를지가 이번 소단원의 주제입니다.

## ■ 1. 첫 비행: 함수를 부르는 순서

드론 프로그램은 항상 같은 뼈대를 가집니다. 드론 객체를 만들고, 연결하고, 날고, 내리고, 연결을 닫습니다. 가운데 '날고'만 바뀔 뿐 앞뒤는 매번 같습니다.

**예제 1-16. 첫 비행**

In [ ]:
# 예제 1-16. 첫 비행
from drone_sim import *        # 실기체: from codrone_edu.drone import *

drone =        # 드론 객체 만들기
# 조종기와 연결

# 이륙

# 3초 동안 제자리 비행

# 착륙

# 연결 닫기


실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 착륙
[가상] 연결을 닫았습니다.
```

한 줄씩 뜯어 봅시다. drone.takeoff()는 세 조각으로 이루어져 있습니다.

| 조각 | 뜻 |
|---|---|
| drone | 명령을 받을 드론 객체. Drone()으로 만들어 이름표를 붙여 둔 것입니다 |
| . (점) | '\~의'. drone이 가진 기능 가운데서 고른다는 표시입니다 |
| takeoff() | 함수 이름과 괄호. 괄호를 붙여야 비로소 실행됩니다 |

<sub>표 1-9. 함수 호출의 구조</sub>

코드론에듀는 이륙 명령을 받으면 바닥에서 평균 80 cm 높이까지 올라간 뒤 제자리에 머물습니다. hover(3)의 3처럼 괄호 안에 넣는 값을 <b>인자</b>라고 합니다. hover는 '몇 초 동안'을 인자로 받습니다.

> **짚고 가기 · pair()를 빠뜨리면?**  
> 연결하지 않고 takeoff()를 부르면 가상 드론은 다음 오류를 내고 멈춥니다. 빨간 글자가 나오면 마지막 줄부터 읽습니다. 무엇이 잘못됐는지는 대개 마지막 줄에 있습니다.  
> NotConnectedError: 드론에 연결되어 있지 않습니다. drone.pair() 를 먼저 실행하세요.

> **안전 수칙 · close()는 꼭 씁니다.**  
> close()는 컴퓨터와 조종기 사이의 연결을 닫습니다. 이 줄이 없으면 다음에 코드를 실행할 때 이전 연결을 찾느라 연결하지 못할 수 있습니다.

## ■ 2. 인자로 비행 조절하기

앞으로 가는 함수는 move\_forward()입니다. 이 함수는 인자를 세 개까지 받습니다.

| 인자 | 뜻 | 안 적으면 |
|---|---|---|
| distance | 갈 거리 | 꼭 적어야 합니다 |
| units | 단위: "cm", "m", "in", "ft" | "cm" |
| speed | 속도(m/s), 최대 2 | 0.5 |

<sub>표 1-10. move\_forward()의 인자</sub>

인자를 넘기는 방법은 두 가지입니다. move\_forward(50, "cm")처럼 순서대로 넣거나, move\_forward(50, speed=1.0)처럼 이름을 붙여 넣습니다. 이름을 붙이면 중간 인자를 건너뛸 수 있습니다. 앞의 것을 <b>위치 인자</b>, 뒤의 것을 <b>키워드 인자</b>라고 합니다.

**예제 1-17. 단위와 속도 바꾸기**

In [ ]:
# 예제 1-17. 단위와 속도 바꾸기
from drone_sim import *

drone = Drone()
drone.pair()
drone.takeoff()

# 50 cm, 기본 속도 0.5 m/s

# 단위를 m 로: 역시 50 cm

# 속도만 키워드로 바꾸기

drone.move_left(20, "mm")             # 없는 단위를 넣으면?

print("앞뒤 위치:", drone.get_pos_x(), "cm")
drone.land()
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 50cm
[가상] 앞으로 0.5m
[가상] 뒤로 30cm
Error: Not a valid unit.
앞뒤 위치: 70.0 cm
[가상] 착륙
[가상] 연결을 닫았습니다.
```

50 + 50 − 30 = 70입니다. 왼쪽 20 mm는 어디로 갔을까요? 드론은 "mm"를 모릅니다. 그래서 오류 문구 한 줄만 남기고 아무것도 하지 않았습니다.

> **짚고 가기 · 멈추지 않는 오류가 더 위험합니다.**  
> pair()를 빠뜨린 오류는 프로그램을 멈춰 주지만, 단위 오류는 한 줄만 출력하고 다음 줄로 넘어갑니다. 실기체에서는 그동안 드론이 제자리에 떠 있고, 그 뒤의 명령은 계획과 다른 자리에서 실행됩니다. 실행 결과에 Error가 한 줄이라도 보이면 실기체로 넘어가지 않습니다. 속도도 같습니다. speed=5를 넣어도 오류 없이 2 m/s로 줄여서 납니다.

> **짚고 가기 · 실기체에서는 70.0이 나오지 않습니다.**  
> 같은 코드를 실기체로 돌리면 62.47 cm처럼 어긋난 값이 나오기도 합니다. get\_pos\_x()는 줄자로 잴 거리가 아닙니다. 드론 밑에 달린 센서가 바닥 무늬가 얼마나 지나갔는지 보고 추정한 값입니다. 그래서 무늬가 없는 바닥이나 빛을 반사하는 바닥에서는 오차가 커집니다. 또 move\_forward()는 '거리 ÷ 속도'만큼 기다린 뒤 다음 줄로 넘어가므로, 기체가 덜 갔거나 더 갔어도 그대로 진행합니다.  
> 실기체에서 위치를 읽을 때는 drone.hover(1)로 멈춘 뒤 읽고, 줄자로 잴 거리와 비교합니다. 센서 값은 한 번 읽은 값보다 여러 번 읽은 경향을 믿습니다.

## ■ 3. 반복문과 함께: 다각형 비행

1-4에서 배운 for문에 드론 함수를 넣으면 도형을 그리며 날 수 있습니다. 변의 수가 n이면 한 꼭짓점에서 도는 각도는 360 / n입니다. 사각형은 90도, 육각형은 60도입니다. 칠각형은 어떻게 될까요?

**예제 1-18. 칠각형 비행**

In [ ]:
# 예제 1-18. 칠각형 비행
from drone_sim import *

drone = Drone()
drone.pair()
drone.takeoff()

sides = 7                 # 칠각형
# 한 번에 돌 각도 = 360 ÷ 변의 수
angle =
print("계산한 회전 각도:", angle)

# 변의 수만큼 반복

    drone.move_forward(30)
    # 계산한 각도만큼 오른쪽으로 회전
    

print("처음 방향과의 차이:", drone.get_angle_z(), "도")
drone.land()
drone.close()

실행 결과 (가운데 다섯 번 반복 생략)

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
계산한 회전 각도: 51.42857142857143
[가상] 앞으로 30cm
[가상] 오른쪽으로 51도 회전
...
[가상] 앞으로 30cm
[가상] 오른쪽으로 51도 회전
처음 방향과의 차이: 3.0 도
[가상] 착륙
[가상] 연결을 닫았습니다.
```

코드는 51.43도를 주었는데 드론은 51도만 돌았습니다. 코드론에듀의 회전 함수는 각도에서 소수점 아래를 버리기 때문입니다. 한 번에 0.43도씩, 일곱 번이면 3도가 모자랍니다. 51 × 7 = 357이므로, 드론은 한 바퀴를 다 돌지 못하고 처음보다 3도 왼쪽을 보며 끝났습니다.

> **한 걸음 더 · 작은 오차는 쌓입니다.**  
> 3도는 작아 보입니다. 그러나 칠각형을 열 번 돌면 30도가 되고, 그 방향으로 5 m를 날면 옆으로 2.5 m 빗나갑니다. 마지막 회전에서만 turn\_right(360 - 51 \* 6)으로 54도를 돌면 합이 정확히 360도가 됩니다.

## ■ 4. 거리 대신 힘과 시간으로 움직이기

move\_forward()는 '얼마나 갈지'를 정하면 드론이 알아서 멈춥니다. 그런데 조종기로 날릴 때 우리는 거리를 정하지 않습니다. 스틱을 얼마나 세게, 얼마나 오래 밀지를 정합니다. 코드로도 같은 일을 할 수 있습니다.

| 함수 | 하는 일 | 값의 범위 |
|---|---|---|
| set\_pitch(힘) | 앞(+)·뒤(−)로 기울이는 힘 | −100 \~ 100 |
| set\_roll(힘) | 오른쪽(+)·왼쪽(−)으로 기울이는 힘 | −100 \~ 100 |
| set\_yaw(힘) | 왼쪽(+)·오른쪽(−)으로 도는 힘 | −100 \~ 100 |
| set\_throttle(힘) | 위(+)·아래(−)로 가는 힘 | −100 \~ 100 |
| move(초) | 정해 둔 힘을 몇 초 동안 실행 | 0보다 큰 수 |

<sub>표 1-11. 비행 변수를 정하는 함수</sub>

set\_으로 시작하는 함수는 값을 저장만 하고 드론을 움직이지 않습니다. move()를 불러야 비로소 날아갑니다. 저장한 값은 지우기 전까지 남아 있으므로, 다 쓰면 set\_pitch(0)처럼 0으로 돌려 둡니다.

**예제 1-19. 바람이 불 때와 불지 않을 때**

In [ ]:
# 예제 1-19. 바람이 불 때와 불지 않을 때
from drone_sim import *

drone = Drone()
drone.pair()

for wind in [0, 3]:
    drone.sim_set_wind(wind)          # 가상 전용: 풍속(m/s)
    drone.takeoff()                   # 이륙하면 위치가 0 으로 돌아간다
    # 앞으로 기울이는 힘 30 %
    
    # 1초 동안
    
    # 힘을 다시 0 으로
    
    drone.hover(1)
    print(f"풍속 {wind} m/s → 앞 {drone.get_pos_x()} cm, 옆 {drone.get_pos_y()} cm")
    drone.land()

drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
풍속 0 m/s → 앞 23.25 cm, 옆 0.0 cm
[가상] 착륙
[가상] 이륙. 고도 약 80 cm
풍속 3 m/s → 앞 23.25 cm, 옆 -26.0 cm
[가상] 착륙
[가상] 연결을 닫았습니다.
```

두 비행 모두 앞으로는 23 cm쯤 갔습니다. 달라진 것은 옆입니다. 옆 위치의 −는 오른쪽입니다. 바람이 불자 드론은 아무 명령도 받지 않았는데 오른쪽으로 26 cm 밀렸습니다. 코드에는 '옆으로 가라'는 줄이 한 줄도 없습니다. 비행이 끝난 뒤 위치를 읽어 보고서야 밀린 것을 알아차릴 수 있었습니다.

> **짚고 가기 · 가상 드론의 바람은 모형입니다.**  
> 26 cm는 가상 드론이 정한 계산식에서 나온 값이지, 코드론에듀를 풍속 3 m/s에서 재 값이 아닙니다. 이 예제에서 믿을 수 있는 것은 '바람이 있으면 명령하지 않은 쪽으로 밀린다'는 경향뿐입니다. 실제 풍속과 비행의 관계는 Ⅱ단원에서 기상 자료로 다시 다룹니다.

## ■ 5. 반환값으로 상태 확인하기

지금까지 부른 함수는 드론에게 '하라'고 시키는 함수였습니다. get\_으로 시작하는 함수는 드론에게 '알려 달라'고 묻습니다. 함수가 돌려주는 답을 <b>반환값</b>이라고 합니다. 반환값은 변수에 담거나 if문에서 바로 비교할 수 있습니다.

| 함수 | 돌려주는 값 |
|---|---|
| get\_battery() | 배터리 잔량(%), 정수 |
| get\_height() | 바닥에서의 높이(cm) |
| get\_pos\_x(), get\_pos\_y() | 이륙한 자리에서 앞뒤·좌우로 움직인 거리(cm) |
| get\_angle\_z() | 처음 방향에서 돈 각도(도, 왼쪽 +) |

<sub>표 1-12. 상태를 알려 주는 함수</sub>

**예제 1-20. 배터리를 확인하며 구간 비행하기**

In [ ]:
# 예제 1-20. 배터리를 확인하며 구간 비행하기
from drone_sim import *

drone = Drone()
drone.sim_set_battery(38)             # 가상 전용: 배터리를 38 % 로 맞춰 둔다
drone.pair()

plan = [120, 90, 150, 110, 130]       # 구간별 비행 거리(cm)
LIMIT = 36                            # 이 아래로 떨어지면 복귀(%)

# 드론에게 배터리 잔량 묻기
battery =
print("비행 전 배터리:", battery, "%")

# 만약 배터리가 30보다 적으면

    print("배터리가 부족하여 비행을 취소합니다.")
else:
    drone.takeoff()
    for n, distance in enumerate(plan, 1):
        # 만약 지금 배터리가 LIMIT보다 적으면
        
            print(f"배터리 {drone.get_battery()} %: {n}번 구간을 남기고 착륙합니다.")
            # 반복을 멈춘다
            
        # 이 구간의 거리만큼 앞으로
        
        print(f"  {n}번 구간 완료, 높이 {drone.get_height()} cm, 배터리 {drone.get_battery()} %")
    drone.land()

drone.sim_save_log()                  # 가상 전용: 비행 기록을 CSV 로 저장
drone.close()

실행 결과

```
[가상] 드론에 연결되었습니다. 배터리 38 %
비행 전 배터리: 38 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 120cm
  1번 구간 완료, 높이 78.58 cm, 배터리 36 %
[가상] 앞으로 90cm
  2번 구간 완료, 높이 79.46 cm, 배터리 36 %
[가상] 앞으로 150cm
  3번 구간 완료, 높이 81.18 cm, 배터리 35 %
배터리 35 %: 4번 구간을 남기고 착륙합니다.
[가상] 착륙
[가상] 비행 기록 8줄을 flight_log.csv 에 저장했습니다.
[가상] 연결을 닫았습니다.
```

enumerate(plan, 1)은 리스트의 값과 함께 1부터 시작하는 번호를 꺼내 줍니다. 1번과 2번 구간이 끝났을 때 배터리는 둘 다 36%로 보입니다. 90 cm를 날았는데 왜 그대로일까요? get\_battery()는 소수점 아래를 버린 정수를 돌려주기 때문입니다. 가상 드론 안에서는 36.6%에서 36.1%로 줄었지만 둘 다 36으로 표시됐습니다. 배터리는 1% 단위로만 보이므로 복귀 기준에는 여유를 두는 것이 안전합니다.

높이도 살펴봅시다. 이륙 높이는 80 cm인데 78.58, 81.18처럼 매번 조금씩 다릅니다. 드론이 흔들리기도 하고 센서에도 오차가 있기 때문입니다. 저장한 flight\_log.csv는 Ⅱ단원에서 pandas로 다시 열어 봅니다.

## ■ 6. 파이참으로 실기체 날리기

가상 드론에서 끝까지 오류 없이 돌아갔다면 이제 실기체로 날려 봅니다. 코랩은 구글 서버에서 돌아가서 우리 PC에 꽂은 조종기를 볼 수 없습니다. 그래서 실기체는 내 PC에서 돌아가는 파이참(PyCharm)으로 날립니다. 아래 절차는 로보링크의 코드론에듀 파이썬 레슨 0.1·0.2·1.1을 따릅니다.

**① 파이썬과 파이참 설치**

1. 파이썬 홈페이지(python.org)에서 파이썬 3.11을 내려받아 설치합니다. 학교 컴퓨터처럼 권한이 제한된 계정이라면 관리자 권한이 필요합니다.
2. 제트브레인스 홈페이지에서 파이참을 내려받아 설치합니다.

> **짚고 가기 · 파이썬 3.12 이상을 쓰면?**  
> 로보링크 레슨에 따르면 파이썬 3.12 이상에서는 No Module named 'pkg\_resources' 오류가 날 수 있습니다. 이때는 3.11 이하 버전을 쓰거나, 명령 프롬프트(cmd)에서 pip install setuptools를 실행합니다.

**② 파이참 환경 설정과 라이브러리 설치**

1. 파이참에서 새 프로젝트를 만들고, 인터프리터(파이썬 번역기)로 설치한 파이썬 3.11을 고릅니다.
2. 테스트 프로그램 print("Hello, world!")를 실행해 파이썬이 제대로 설치되었는지 확인합니다. 출력은 화면 아래의 Run 창에 나타납니다.
3. File → Settings(맥은 PyCharm → Settings) → Project → Python Interpreter에서 + 단추를 누르고, codrone-edu를 검색해 설치합니다.
4. main.py에 아래 한 줄을 적고 실행합니다. 오류가 없으면 라이브러리가 제대로 설치된 것입니다. 아직 드론을 연결할 필요는 없습니다.

```python
from codrone_edu.drone import *
```

main.py가 만들어지지 않았다면 프로젝트 이름을 마우스 오른쪽 단추로 누르고 New → Python File을 고릅니다.

**③ 조종기와 연결**

1. 드론에 배터리를 끼우고 드론과 조종기의 전원을 켭니다. 두 기기는 가까이 있으면 스스로 짝을 맺습니다.
2. 드론과 함께 든 USB 케이블로 조종기를 PC에 연결합니다.
3. 조종기가 링크(LINK) 상태가 되었는지 확인합니다. 코드는 이 상태에서만 드론에 전달됩니다.

> **짚고 가기 · LINK가 뜨지 않으면**  
> 충전만 되고 데이터는 주고받지 못하는 케이블일 수 있습니다. 드론과 함께 든 케이블로 바꿔 꽂습니다.

**④ 테스트 코드 실행**

첫 코드에서는 드론을 띄우지 않습니다. 불빛을 켜고 소리를 내 연결만 확인합니다. 코드를 입력하고 파이참 위쪽의 초록색 삼각형을 누릅니다. 코드의 아무 곳이나 마우스 오른쪽 단추로 눌러 Run을 골라도 됩니다.

**예제 1-21. 연결 확인하기** — 파이참 · 실기체

코랩에서는 실행하지 않습니다. 파이참에서 실기체로 실행합니다.

```python
from codrone_edu.drone import *

drone = Drone()
drone.pair()
drone.set_drone_LED(255, 255, 255, 255)   # 흰색 불빛, 밝기 최대
drone.drone_buzzer(Note.C4, 1000)        # 도 1초
drone.drone_buzzer(Note.D4, 1000)        # 레 1초
drone.drone_buzzer(Note.E4, 1000)        # 미 1초
drone.close()
```

드론에 흰 불이 켜지고 도·레·미가 들리면 성공입니다. 오류가 나면 드론과 조종기가 제대로 연결되었는지 다시 확인합니다.

**⑤ 페어링과 출력**

프로그램이 어디까지 실행되었는지 알고 싶을 때 print()를 사이사이에 넣습니다. 멈춘 곳 바로 앞의 print까지만 출력되므로, 어느 줄에서 문제가 생겼는지 바로 보입니다.

**예제 1-22. 페어링 단계 출력하기** — 파이참 · 실기체

코랩에서는 실행하지 않습니다. 파이참에서 실기체로 실행합니다.

```python
from codrone_edu.drone import *

print("드론 객체 만드는 중")
drone = Drone()
print("페어링 준비")
drone.pair()
print("페어링 성공!")
drone.close()
```

> **짚고 가기 · import 줄을 빠뜨리면**  
> 파이참은 Drone 아래에 빨간 물결선을 그어 알려 줍니다. Drone이 무엇인지 파이썬이 모르기 때문입니다. 파일마다 첫 줄에 from codrone\_edu.drone import \*를 씁니다.

**⑥ 가상에서 짠 코드를 실기체로**

코랩에서 만든 예제 1-16을 파이참으로 가져와 flight\_test.py로 저장합니다. 파일 이름에는 빈칸 대신 밑줄(\_)을 씁니다. 바꿀 것은 두 가지뿐입니다.

1. 첫 줄 from drone\_sim import \*를 from codrone\_edu.drone import \*로 바꿉니다.
2. sim\_으로 시작하는 줄을 지웁니다. 실기체 라이브러리에는 이 함수가 없어서 AttributeError가 납니다.

실기체에서는 가상 드론에 없던 일이 생깁니다.

| 상황 | 원인 | 대처 방법 |
|---|---|---|
| pair()에서 멈춤 | USB 케이블, 조종기 전원, LINK 상태 아님 | 케이블과 전원을 확인하고 다시 실행 |
| 이륙하지 않음 | 배터리 부족 | get\_battery()로 잔량을 확인하고 충전 |
| 한쪽으로 치우쳐 날아감 | 프로펠러 파손, 트림 틀어짐 | 즉시 착륙시키고 기체 점검 |
| 명령에 반응하지 않음 | 통신 범위를 벗어남 | 비상 정지 |
| 예상보다 멀리 감 | 단위를 잘못 넣음(m와 cm) | 단위 인자 확인 |

<sub>표 1-13. 비행 중 오류 상황과 대처 방법</sub>

> **안전 수칙 · 비상 정지는 두 가지입니다.**  
> 코드에서는 drone.emergency\_stop()이 모터를 곧바로 멈춥니다. 조종기에서는 L1을 누른 채 왼쪽 스틱을 아래로 당깁니다(코드론에듀 사용 설명서 v3.2). 비상 정지는 공중에서 모터를 끄므로 드론이 그대로 떨어집니다. 사람이나 물건에 부딪히기 직전에만 씁니다.
> 실기체 실습 3원칙
> 1. 작게 시작합니다. 거리와 속도는 작은 값부터 조금씩 늘립니다.
> 2. 손은 비상 정지에 둡니다. 날고 있는 동안 한 사람은 조종기를 들고 있습니다.
> 3. 한 번에 하나만 바꿉니다. 여러 값을 한꺼번에 바꾸면 무엇 때문에 달라졌는지 알 수 없습니다.

> **AI와 함께 · AI가 알려 준 함수가 진짜 있는지 확인합니다.**  
> 생성형 AI에게 코드론 코드를 부탁하면 drone.fly\_to(), drone.rotate()처럼 있을 법한 이름을 지어내는 일이 있습니다. 가상 드론에서 먼저 실행하면 없는 함수는 AttributeError로 바로 드러납니다. 오류 문구를 AI에게 보여 주며 "이 오류가 무슨 뜻인지 설명해 줘"라고 묻는 것은 좋은 사용법입니다. 다만 고친 코드는 다시 가상 드론에서 돌려 본 뒤에 실기체로 옮깁니다.

## ■ 7. 동작을 함수로 묶어 다시 쓰기

크기만 다른 사각형을 두 번 날리려면 예제 1-11의 for문을 두 번 써야 합니다. 이럴 때 def로 동작 묶음에 이름을 붙여 두면, takeoff()처럼 이름만 불러 몇 번이고 다시 쓸 수 있습니다. 괄호 안의 length는 부를 때마다 넣는 값을 받는 자리입니다.

**예제 1-23. 사각형 비행을 함수로 묶어 다시 쓰기**

In [ ]:
# 예제 1-23. 사각형 비행을 함수로 묶어 다시 쓰기
from drone_sim import *

# fly_square(length) 함수 만들기: 사각형 비행을 함수로 묶기

    # 네 번 반복
    
        # length만큼 앞으로
        
        # 오른쪽으로 90도
        

drone = Drone()
drone.pair()
drone.takeoff()

# 한 변 30 cm로 fly_square 부르기

# 한 변 60 cm로 같은 함수를 다시 부르기


print("제자리로 돌아왔나요?", drone.get_pos_x(), drone.get_pos_y())
drone.land()
drone.close()

실행 결과 (가운데 반복 생략)

```
[가상] 드론에 연결되었습니다. 배터리 100 %
[가상] 이륙. 고도 약 80 cm
[가상] 앞으로 30cm
[가상] 오른쪽으로 90도 회전
...
[가상] 앞으로 60cm
[가상] 오른쪽으로 90도 회전
제자리로 돌아왔나요? 0.0 -0.0
[가상] 착륙
[가상] 연결을 닫았습니다.
```

def fly\_square(length):는 '이제부터 fly\_square라는 이름으로 아래 동작을 묶는다'는 뜻입니다. 들여 쓴 두 줄이 함수의 몸통이고, 이 줄들은 정의할 때가 아니라 fly\_square(30)처럼 부를 때 실행됩니다. 함수 안의 drone도 부르는 순간에 찾기 때문에, drone = Drone()이 함수 정의보다 아래에 있어도 됩니다. 같은 함수를 30과 60으로 두 번 불러 크기만 다른 사각형을 두 번 그렸고, 두 번 모두 제자리로 돌아왔습니다. 값을 돌려주는 함수나 기본값을 가진 함수처럼 함수를 더 다양하게 만드는 법은 2학기 'AI 드론 지능형 제어'에서 배웁니다.

### ====== 해 보기 ======

**우리 모둠의 비행 임무 설계하기**

In [ ]:
# 해 보기: 필요한 코드나 답을 여기에 적어 봅시다.


모둠별로 실습 공간에 맞는 비행 임무를 설계하고 프로그램으로 만들어 봅시다.

1. 종이에 경로를 그리고, 구간마다 거리(cm)와 회전 각도를 적습니다. 각도는 정수로 정합니다.
2. 거리를 리스트로 만들고 for문으로 프로그램을 작성합니다. 같은 동작이 두 번 넘게 나오면 예제 1-23처럼 함수로 묶어 다시 씁니다.
3. 코랩의 가상 드론에서 실행하고, 끝난 뒤 get\_pos\_x(), get\_pos\_y(), get\_angle\_z()를 출력해 의도한 자리인지 확인합니다.
4. sim\_set\_wind(2)를 넣어 다시 실행하고, 옆으로 얼마나 밀렸는지 기록합니다.
5. 파이참으로 옮겨 첫 줄을 바꾸고 sim\_ 줄을 지운 뒤, 교사의 확인을 받아 실기체로 날립니다.
6. 도착한 자리를 줄자로 재어 가상 드론의 결과와 비교하고, 차이가 생긴 까닭을 모둠에서 이야기합니다.

### 핵심 정리

- 드론 프로그램은 만들기 → pair() → takeoff() → 비행 → land() → close() 순서를 따릅니다.
- 인자로 거리·단위·속도·각도를 바꾸며, 단위 오류는 프로그램을 멈추지 않으므로 실행 결과를 끝까지 읽습니다.
- get\_으로 시작하는 함수는 드론의 상태를 반환값으로 돌려주며, if문과 함께 쓰면 상황에 따라 비행을 바꿀 수 있습니다.
- 가상 드론에서 검증한 뒤 파이참에서 첫 줄만 바꿔 실기체로 옮기고, 작게 시작해 한 번에 하나씩 바꿉니다.

<sub>스스로 확인하기는 교재의 문제를 풉니다. 정답과 해설은 부록 3에 있습니다.</sub>

# Ⅰ단원 마무리

### 배운 내용 한눈에 보기

| 소단원 | 꼭 기억할 것 |
|---|---|
| 1-1 드론의 이해와 코딩 환경 구성 | 드론은 양력·중력·추력·항력 사이의 균형으로 날고, 비행 제어기가 네 모터의 회전 속도를 조절합니다. 예제는 코랩과 가상 드론으로 실행합니다. |
| 1-2 파이썬 기본 문법과 자료형 | 변수는 값에 붙이는 이름표입니다. 값의 종류(int, float, str, bool)에 따라 할 수 있는 일이 달라집니다. |
| 1-3 연산과 입출력 | input()으로 받은 값은 문자열이므로 바꾸어서 계산합니다. 비교·논리 연산의 결과(True, False)가 판단의 근거가 됩니다. |
| 1-4 제어문과 자료구조 | if로 갈래를 나누고 for·while로 되풀이합니다. 리스트·튜플·딕셔너리에 경유지와 비행 기록을 담습니다. |
| 1-5 드론 기초 제어 | 함수를 부를 때 인자로 거리·단위·속도를 정하고, get\_으로 시작하는 함수의 반환값으로 상태를 확인합니다. 가상 드론에서 검증한 코드만 실기체로 옮깁니다. |

### 스스로 점검하기

단원을 시작할 때 세운 목표를 얼마나 이루었는지 표시해 봅니다. ○는 '혼자 할 수 있다', △는 '책을 보면 할 수 있다', ✕는 '아직 어렵다'입니다.

| 점검 내용 | ○ △ ✕ |
|---|---|
| 드론의 구성 요소와 비행 원리를 설명하고 안전수칙을 지킬 수 있다. |  |
| 변수와 자료형, 연산과 입출력을 활용하여 드론의 상태를 계산하고 나타낼 수 있다. |  |
| 조건문·반복문과 자료구조를 활용하여 상황에 따라 달라지는 비행 프로그램을 작성할 수 있다. |  |
| 드론 라이브러리의 함수를 활용하여 드론의 기본 동작을 제어하는 프로그램을 작성하고 실행할 수 있다. |  |

△나 ✕에 표시한 항목은 관련 소단원의 핵심 정리와 예제로 돌아가 다시 살펴봅니다.